# AIFlow — audited SDXL image worker
Use a T4 GPU, Python 3.12. The reviewed bundle contains hash-locked wheel requirements and immutable safetensors model inventories. No ComfyUI, custom nodes, git installs, pickle weights, or Drive-hosted Python code.

Upload only the reviewed `aiflow-image-worker.zip` produced from this checkout. The package audit must approve the exact requirements bytes before installation. The preinstalled Colab Torch 2.11.0+cu128 has a documented narrow advisory exception: this worker never invokes Torch JIT/compile or loads pickle/remote Python. A changed runtime fails closed.

The service binds localhost and authenticates every API route with a session Bearer token. The audited tunnel is public transport; every forwarded API route remains protected by the private Bearer token. Never publish the token in notebook output. Colab local `/content` storage is temporary; use an explicitly selected durable storage folder for reconnect persistence. This notebook does not mount Google Drive automatically.


In [ ]:
from google.colab import files
from pathlib import Path, PurePosixPath
import zipfile

uploaded = files.upload()
if set(uploaded) != {'aiflow-image-worker.zip'}:
    raise RuntimeError('Upload exactly the reviewed aiflow-image-worker.zip')
import hashlib
if hashlib.sha256(Path('aiflow-image-worker.zip').read_bytes()).hexdigest() != '1d3e7cec00306f757d14d10e95dee298fcac14f4767ccc0e8d6f76c6812fffea':
    raise RuntimeError('Bundle differs from the reviewed notebook SHA256')
CODE_ROOT = Path('/content/aiflow-image-code')
CODE_ROOT.mkdir(exist_ok=True)
allowed = {'__init__.py','__main__.py','app.py','bootstrap.py','engine.py','models.py','model-lock.json',
           'requirements-lock.txt','audit-approval.json','README.md'}
with zipfile.ZipFile('aiflow-image-worker.zip') as archive:
    total = 0
    for entry in archive.infolist():
        name = PurePosixPath(entry.filename)
        if len(name.parts) != 2 or name.parts[0] != 'image_worker' or name.name not in allowed:
            raise RuntimeError('Unexpected file in reviewed bundle')
        if entry.is_dir() or entry.file_size > 2_000_000:
            raise RuntimeError('Invalid bundle member')
        total += entry.file_size
        if total > 3_000_000:
            raise RuntimeError('Bundle is too large')
        destination = CODE_ROOT / 'image_worker' / name.name
        destination.parent.mkdir(exist_ok=True)
        destination.write_bytes(archive.read(entry))
del uploaded
print('Worker bundle extracted under /content')


In [ ]:
# Isolate audited Python packages, inherit only the reviewed Colab GPU Torch runtime.
import subprocess, sys, os, json, hashlib
VENV = Path('/content/aiflow-image-venv')
subprocess.run([sys.executable, '-m', 'venv', '--system-site-packages', '--without-pip', str(VENV)], check=True)
PYTHON = VENV / 'bin/python'
# Ubuntu Colab may omit ensurepip; inherited pip installs to this venv's sys.prefix.
subprocess.run([str(PYTHON), '-c', "import sys; from pathlib import Path; assert sys.prefix != sys.base_prefix and Path(sys.prefix).resolve() == Path('/content/aiflow-image-venv').resolve(), 'Wrong installation target'"], check=True)
env = dict(os.environ, PYTHONPATH=str(CODE_ROOT), HF_HUB_DISABLE_TELEMETRY='1', DO_NOT_TRACK='1', HF_HUB_DISABLE_IMPLICIT_TOKEN='1', DIFFUSERS_DISABLE_REMOTE_CODE='1')
subprocess.run([str(PYTHON), '-m', 'image_worker.bootstrap'], env=env, check=True)
subprocess.run([str(PYTHON), '-c', 'from image_worker.bootstrap import verify_runtime; import json; print(json.dumps(verify_runtime()))'],
               env=env, check=True)


In [ ]:
# Download exactly 21 approved config/tokenizer/safetensors files (about 10 GB), then SHA256 verify.
MODEL_ROOT = Path('/content/aiflow-image-models')
subprocess.run([str(PYTHON), '-m', 'image_worker', 'download', '--model-root', str(MODEL_ROOT)], env=env, check=True)


## Start the authenticated API
Supply a fresh private token with at least 32 random ASCII bytes; copy the same token into AIFlow's image-worker connection form. Use Colab Secrets `AIFLOW_IMAGE_WORKER_TOKEN` or the hidden prompt. Do not store it in Drive or a manifest.

`STORAGE_ROOT` defaults to `/content/aiflow-image-jobs`. If you choose durable storage, select only a trusted non-code folder: restarting retains completed jobs and marks unfinished jobs failed; it never silently regenerates them.


In [ ]:
if 'worker' in globals() and worker.poll() is None:
    print('Worker already running; keeping the existing process and session token')
else:
    from google.colab import userdata
    from getpass import getpass
    try:
        session_token = userdata.get('AIFLOW_IMAGE_WORKER_TOKEN')
    except Exception:
        session_token = getpass('Private image-worker Bearer token (32+ random ASCII bytes): ')
    if not session_token or len(session_token.encode()) < 32 or not session_token.isascii() or any(c.isspace() for c in session_token):
        raise RuntimeError('Invalid session token')
    STORAGE_ROOT = Path('/content/aiflow-image-jobs')
    worker_env = dict(env, AIFLOW_IMAGE_WORKER_TOKEN=session_token, HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1')
    # Model initialization takes several minutes. No access logs, public bind, or token output.
    log_file = open('/content/aiflow-image-worker.log', 'w')
    worker = subprocess.Popen([str(PYTHON), '-m', 'image_worker', 'serve', '--model-root', str(MODEL_ROOT),
                               '--storage', str(STORAGE_ROOT)], env=worker_env, stdout=log_file, stderr=log_file)
    print('Worker starting on localhost:8877; run health check after initialization')


In [ ]:
# Run manually once initialization has finished. Connection refusal means it is still loading or failed.
import urllib.request
request = urllib.request.Request('http://127.0.0.1:8877/v1/health',
                                 headers={'Authorization': 'Bearer ' + session_token})
with urllib.request.urlopen(request, timeout=30) as response:
    health = json.load(response)
if health.get('status') != 'ready' or health.get('api_version') != '1':
    raise RuntimeError('Worker is not ready')
print(json.dumps(health, indent=2))


## Optional real GPU smoke without a tunnel
Run All skips batch smoke by default. Set `RUN_SMOKE = True` in the next cell only when you intend to stop the API and run a batch test so only one model uses the GPU. Upload a request JSON matching the API schema: reference mode uses one to three base64 PNG/JPEG references with verified hashes; fictional human text mode uses `generation_mode="text"`, `subject_type="human"`, and `references=[]`. Text mode runs SDXL without IP-Adapter conditioning. A fictional prompt does not guarantee resemblance or copyright clearance. A genuine GPU run writes the PNG and a provenance receipt including model revisions, settings, elapsed time, hardware and output hash. It does not claim face/identity accuracy automatically; inspect the resulting human/pet subject.


In [ ]:
if 'tunnel' in globals() and tunnel.poll() is None:
    print('Tunnel already running; keeping the existing endpoint')
else:
    # Optional audited Cloudflare Quick Tunnel. Every forwarded API route still requires the private Bearer token.
    # Never publish a token in this cell or in its output; enter it only in AIFlow's connection form.
    import urllib.request, hashlib, os
    TUNNEL_BINARY = Path('/content/aiflow-cloudflared')
    TUNNEL_SHA256 = '77e26d8d900e0b8469f416239d14b5f296525fdf79fee6f511ef55609e3fbac2'
    TUNNEL_URL = 'https://github.com/cloudflare/cloudflared/releases/download/2026.9.3/cloudflared-linux-amd64'
    if not TUNNEL_BINARY.exists():
        temporary = TUNNEL_BINARY.with_suffix('.partial')
        size = 0
        with urllib.request.urlopen(TUNNEL_URL, timeout=60) as source, temporary.open('wb') as target:
            while True:
                chunk = source.read(1024 * 1024)
                if not chunk:
                    break
                size += len(chunk)
                if size > 40122749:
                    raise RuntimeError('Tunnel binary exceeds reviewed size')
                target.write(chunk)
        if size != 40122749 or hashlib.sha256(temporary.read_bytes()).hexdigest() != TUNNEL_SHA256:
            raise RuntimeError('Tunnel binary fails reviewed size/hash')
        temporary.replace(TUNNEL_BINARY)
    if hashlib.sha256(TUNNEL_BINARY.read_bytes()).hexdigest() != TUNNEL_SHA256:
        raise RuntimeError('Tunnel binary fails reviewed hash')
    TUNNEL_BINARY.chmod(0o700)
    tunnel_log = open('/content/aiflow-image-tunnel.log', 'w')
    tunnel = subprocess.Popen([str(TUNNEL_BINARY), 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:8877'],
                              stdout=tunnel_log, stderr=tunnel_log)
    print('Tunnel starting; run the next cell once to read its endpoint')


In [ ]:
# This log contains only tunnel diagnostics, not request headers or the session token.
import re
endpoints = re.findall(r'https://[a-z0-9-]+\.trycloudflare\.com', Path('/content/aiflow-image-tunnel.log').read_text())
if not endpoints:
    raise RuntimeError('Tunnel endpoint not yet available; review tunnel log locally')
print('AIFlow worker URL: ' + endpoints[-1])


In [ ]:
RUN_SMOKE = False  # Change to True only for an explicit batch test; Run All keeps the API running.
if RUN_SMOKE:
    # Optional: run instead of the API when testing an unavailable tunnel.
    # If the API was started above, explicitly stop its own process first.
    if 'worker' in globals() and worker.poll() is None:
        worker.terminate()
        worker.wait(timeout=30)
    smoke_uploaded = files.upload()
    if len(smoke_uploaded) != 1:
        raise RuntimeError('Upload one smoke request JSON')
    smoke_request = Path('/content/aiflow-image-smoke-request.json')
    smoke_request.write_bytes(next(iter(smoke_uploaded.values())))
    del smoke_uploaded
    subprocess.run([str(PYTHON), '-m', 'image_worker', 'smoke', '--model-root', str(MODEL_ROOT),
                    '--request', str(smoke_request), '--output', '/content/aiflow-image-smoke.png'],
                   env=dict(env, HF_HUB_OFFLINE='1', TRANSFORMERS_OFFLINE='1'), check=True)
    from IPython.display import display, Image as NotebookImage
    display(NotebookImage(filename='/content/aiflow-image-smoke.png'))
    files.download('/content/aiflow-image-smoke.png')
    files.download('/content/aiflow-image-smoke.json')
else:
    print('Optional GPU smoke skipped; worker and tunnel remain running')
